### **Crear el Volumen**

In [0]:
# Volumen
spark.sql("CREATE CATALOG IF NOT EXISTS proyecto")
spark.sql("CREATE SCHEMA IF NOT EXISTS proyecto.sentimiento")
spark.sql("CREATE VOLUME IF NOT EXISTS proyecto.sentimiento.datos")
print("Sube Reviews.csv a: /Volumes/proyecto/sentimiento/datos/")

Sube Reviews.csv a: /Volumes/proyecto/sentimiento/datos/


### **Capa Bronce**

In [0]:
ruta = "/Volumes/proyecto/sentimiento/datos/Reviews.csv"

# Bronce = datos crudos, sin transformar
df_bronce = (spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("inferSchema", "true")
    .csv(ruta))

# Guardar como tabla Delta (capa Bronce)
tabla_bronce = "proyecto.sentimiento.bronze_reviews"
df_bronce.write.mode("overwrite").saveAsTable("proyecto.sentimiento.bronze_reviews")

print(f"Capa Bronce creada: {df_bronce.count():,} reseñas")
df_bronce.printSchema()

Capa Bronce creada: 568,454 reseñas
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



### **Capa Plata (limpieza + anonimización + etiquetas)**

In [0]:
from pyspark.sql.functions import col, when, length, to_date, from_unixtime

# Leer desde la Capa Bronce
df_bronce = spark.table("proyecto.sentimiento.bronze_reviews")

df_plata = (df_bronce
    # 1. ANONIMIZACIÓN: eliminamos datos personales (confidencialidad - CE2_CD1)
    .drop("UserId", "ProfileName")

    # 2. LIMPIEZA: quitar reseñas sin texto
    .filter(col("Text").isNotNull() & (length(col("Text")) > 0))

    # 3. Convertir el timestamp Unix a fecha legible
    .withColumn("fecha", to_date(from_unixtime(col("Time"))))

    # 4. ETIQUETA de sentimiento (3 clases) desde el Score
    .withColumn("label",
        when(col("Score") <= 2, 0)      # Negativo
        .when(col("Score") == 3, 1)     # Neutro
        .otherwise(2))                  # Positivo (4-5)

    # 5. Etiqueta en texto (para gráficos legibles después)
    .withColumn("sentimiento",
        when(col("Score") <= 2, "Negativo")
        .when(col("Score") == 3, "Neutro")
        .otherwise("Positivo"))
)

# Guardar Capa Plata
df_plata.write.mode("overwrite").saveAsTable("proyecto.sentimiento.silver_reviews")

print(f"Capa Plata creada: {df_plata.count():,} reseñas limpias")
print("\nColumnas finales (sin datos personales):")
print(df_plata.columns)

Capa Plata creada: 568,454 reseñas limpias

Columnas finales (sin datos personales):
['Id', 'ProductId', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'fecha', 'label', 'sentimiento']


Analisis del Desbalanceo

In [0]:
from pyspark.sql.functions import count, round as spark_round

total = df_plata.count()
balance = (df_plata.groupBy("sentimiento", "label")
    .agg(count("*").alias("cantidad"))
    .withColumn("porcentaje", spark_round(col("cantidad") / total * 100, 2))
    .orderBy("label"))

balance.show()

+-----------+-----+--------+----------+
|sentimiento|label|cantidad|porcentaje|
+-----------+-----+--------+----------+
|   Negativo|    0|   82037|     14.43|
|     Neutro|    1|   42640|       7.5|
|   Positivo|    2|  443777|     78.07|
+-----------+-----+--------+----------+



### **Modelo ML** Regresion Logistica
Calculamos los pesos del balanceo

In [0]:
from pyspark.sql.functions import col, when

df_plata = spark.table("proyecto.sentimiento.silver_reviews")
total = df_plata.count()
n_clases = 3

# Peso de cada clase = total / (n_clases * cantidad_de_esa_clase)
# Las clases chicas reciben más peso (el modelo las "toma más en serio")
conteos = {r["label"]: r["cantidad"] for r in
           df_plata.groupBy("label").count().withColumnRenamed("count","cantidad").collect()}

pesos = {k: total / (n_clases * v) for k, v in conteos.items()}
print("Pesos por clase:", pesos)

# Agregar columna de peso a cada fila
df_pesado = df_plata.withColumn("peso",
    when(col("label") == 0, pesos[0])
    .when(col("label") == 1, pesos[1])
    .otherwise(pesos[2]))

Pesos por clase: {2: 0.4269817197977062, 0: 2.3097464152354017, 1: 4.443824265165729}


Validacion con muestra chica

In [0]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# MUESTRA para validar el código sin gastar cuota
df_muestra = df_pesado.sample(fraction=0.05, seed=42)  # ~5% = ~28k filas
print(f"Muestra de prueba: {df_muestra.count():,} filas")

# Pipeline de NLP
tokenizer = Tokenizer(inputCol="Text", outputCol="palabras")
remover   = StopWordsRemover(inputCol="palabras", outputCol="limpias")
hashingTF = HashingTF(inputCol="limpias", outputCol="tf", numFeatures=10000)
idf       = IDF(inputCol="tf", outputCol="features")

# Modelo CON balanceo (weightCol="peso")
lr = LogisticRegression(featuresCol="features", labelCol="label",
                        weightCol="peso", maxIter=20)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Entrenar y evaluar la muestra
train_m, test_m = df_muestra.randomSplit([0.8, 0.2], seed=42)
modelo_m = pipeline.fit(train_m)
pred_m = modelo_m.transform(test_m)

acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred_m)
f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred_m)
print(f"[MUESTRA] Accuracy: {acc:.4f} | F1: {f1:.4f}")

Muestra de prueba: 28,209 filas
[MUESTRA] Accuracy: 0.7753 | F1: 0.7795


Entrenamiento con la data completa y guardado

In [0]:
import time
inicio = time.time()

# Entrenar con TODO el dataset (df_pesado tiene las 568k con peso)
train, test = df_pesado.randomSplit([0.8, 0.2], seed=42)
print(f"Entrenamiento: {train.count():,} | Prueba: {test.count():,}")

# Mismo pipeline validado, ahora sobre todo
modelo_final = pipeline.fit(train)
print(f"Modelo entrenado en {time.time()-inicio:.0f} segundos")

# Guardar el modelo para no re-entrenar nunca más
modelo_final.write().overwrite().save("/Volumes/proyecto/sentimiento/datos/modelo_sentimiento")
print("Modelo guardado en el volumen")

Entrenamiento: 454,804 | Prueba: 113,650
Modelo entrenado en 82 segundos
Modelo guardado en el volumen


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

pred = modelo_final.transform(test)

acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision").evaluate(pred)
rec  = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall").evaluate(pred)

print(f"Accuracy:  {acc:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")

# Matriz de confusión (para ver dónde acierta/falla por clase)
print("\nMatriz de confusión (real vs predicho):")
pred.groupBy("label").pivot("prediction").count().orderBy("label").show()

Accuracy:  0.7685
F1-Score:  0.7969
Precision: 0.8499
Recall:    0.7685

Matriz de confusión (real vs predicho):
+-----+-----+-----+-----+
|label|  0.0|  1.0|  2.0|
+-----+-----+-----+-----+
|    0|11792| 2997| 1624|
|    1| 1796| 5229| 1517|
|    2| 6989|11385|70321|
+-----+-----+-----+-----+



Comparacion con Random Forest

In [0]:
import time
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

inicio = time.time()

# Random Forest no maneja bien vectores enormes (10k features de texto).
# Bajamos numFeatures a 1000 para que sea viable. Con 10k, RF se vuelve lentísimo
tokenizer = Tokenizer(inputCol="Text", outputCol="palabras")
remover   = StopWordsRemover(inputCol="palabras", outputCol="limpias")
hashingTF = HashingTF(inputCol="limpias", outputCol="tf", numFeatures=1000)
idf       = IDF(inputCol="tf", outputCol="features")

rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=30, maxDepth=10, seed=42)

pipeline_rf = Pipeline(stages=[tokenizer, remover, hashingTF, idf, rf])

# Mismo split (train/test) que la Regresión Logística
train, test = df_pesado.randomSplit([0.8, 0.2], seed=42)

modelo_rf = pipeline_rf.fit(train)
print(f"Random Forest entrenado en {time.time()-inicio:.0f} segundos")

# Evaluar
pred_rf = modelo_rf.transform(test)
acc_rf = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred_rf)
f1_rf  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred_rf)

print(f"\n=== RANDOM FOREST ===")
print(f"Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f}")

print("\nMatriz de confusión RF:")
pred_rf.groupBy("label").pivot("prediction").count().orderBy("label").show()

Random Forest entrenado en 153 segundos

=== RANDOM FOREST ===
Accuracy: 0.7817 | F1: 0.6872

Matriz de confusión RF:
+-----+----+----+-----+
|label| 0.0| 1.0|  2.0|
+-----+----+----+-----+
|    0| 116|NULL|16297|
|    1|NULL|  30| 8512|
|    2|NULL|NULL|88695|
+-----+----+----+-----+



### **Capa De Oro**

In [0]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, when

# Cargar el modelo guardado (regresion logistica)
modelo = PipelineModel.load("/Volumes/proyecto/sentimiento/datos/modelo_sentimiento")

# Aplicar el modelo a toda la Capa Plata
df_plata = spark.table("proyecto.sentimiento.silver_reviews")
df_pred = modelo.transform(df_plata)

# Traducir la predicción numérica a texto
df_oro = df_pred.withColumn("sentimiento_predicho",
    when(col("prediction") == 0, "Negativo")
    .when(col("prediction") == 1, "Neutro")
    .otherwise("Positivo")
).select("Id", "ProductId", "Score", "fecha", "sentimiento",
         "sentimiento_predicho", "Summary", "Text",
         "HelpfulnessNumerator", "HelpfulnessDenominator")

# Guardar tabla de detalle (Capa Oro base)
df_oro.write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_reviews")
print("Capa Oro (detalle) creada")

Capa Oro (detalle) creada


In [0]:
from pyspark.sql.functions import count, avg, round as r, year

df_oro = spark.table("proyecto.sentimiento.gold_reviews")

# 1. Sentimiento por producto (los productos problemáticos)
gold_productos = (df_oro.groupBy("ProductId")
    .agg(count("*").alias("total_reviews"),
         r(avg("Score"), 2).alias("score_promedio"),
         count(when(col("sentimiento")=="Negativo", True)).alias("negativas"))
    .withColumn("pct_negativas", r(col("negativas")/col("total_reviews")*100, 1))
    .filter(col("total_reviews") >= 10))   # solo productos con reviews suficientes
gold_productos.write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_por_producto")

# 2. Sentimiento por año (tendencia temporal)
gold_tiempo = (df_oro.withColumn("anio", year("fecha"))
    .groupBy("anio", "sentimiento").count()
    .orderBy("anio"))
gold_tiempo.write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_por_anio")

# 3. Distribución general de sentimiento (para el gráfico circular)
gold_dist = df_oro.groupBy("sentimiento").count()
gold_dist.write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_distribucion")

print("Tablas agregadas de la Capa Oro creadas")
gold_dist.show()

Tablas agregadas de la Capa Oro creadas
+-----------+------+
|sentimiento| count|
+-----------+------+
|   Negativo| 82037|
|     Neutro| 42640|
|   Positivo|443777|
+-----------+------+



In [0]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from pyspark.sql.functions import explode, col, count, lower, length, regexp_replace

# Reseñas negativas
negativas = spark.table("proyecto.sentimiento.gold_reviews").filter(col("sentimiento")=="Negativo")

# 1. Limpiar HTML (<br />, etc.) y caracteres raros del texto
neg_limpio_txt = negativas.withColumn("Text",
    regexp_replace(lower(col("Text")), "<[^>]*>|/>|<br|[^a-z ]", " "))

# 2. Tokenizar
tok = Tokenizer(inputCol="Text", outputCol="palabras")
neg_tok = tok.transform(neg_limpio_txt)

# 3. Stopwords estándar + palabras genéricas que no aportan al "problema"
sw_default = StopWordsRemover.loadDefaultStopWords("english")
sw_extra = ["like","good","taste","food","coffee","product","even","really","much",
            "first","made","amazon","little","make","also","time","know","used",
            "one","get","would","could","tried","bought","thought","love","flavor",
            "buy","br","dont","didnt","im","ive","it","this","that","these"]
sw = StopWordsRemover(inputCol="palabras", outputCol="limpias",
                      stopWords=sw_default + sw_extra)
neg_final = sw.transform(neg_tok)

# 4. Contar
palabras = (neg_final.select(explode("limpias").alias("palabra"))
    .filter(length(col("palabra")) > 3)
    .groupBy("palabra").agg(count("*").alias("frecuencia"))
    .orderBy(col("frecuencia").desc()))

print("Top 25 palabras en reseñas NEGATIVAS (limpio):")
palabras.show(25, truncate=False)

palabras.limit(50).write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_palabras_negativas")

Top 25 palabras en reseñas NEGATIVAS (limpio):
+------------+----------+
|palabra     |frecuencia|
+------------+----------+
|better      |10355     |
|water       |9780      |
|didn        |8960      |
|great       |8905      |
|chocolate   |8695      |
|sugar       |8590      |
|price       |8355      |
|never       |8340      |
|something   |8167      |
|order       |8151      |
|think       |8013      |
|back        |7992      |
|ordered     |7990      |
|well        |7978      |
|tastes      |7962      |
|money       |7743      |
|ingredients |7634      |
|still       |7626      |
|find        |7607      |
|disappointed|7281      |
|products    |7259      |
|brand       |7085      |
|give        |6852      |
|since       |6660      |
|stuff       |6572      |
+------------+----------+
only showing top 25 rows


In [0]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, NGram
from pyspark.sql.functions import explode, col, count, lower, length, regexp_replace

# Reseñas negativas
negativas = spark.table("proyecto.sentimiento.gold_reviews").filter(col("sentimiento")=="Negativo")

# 1. Limpiar HTML y dejar solo letras
neg_txt = negativas.withColumn("Text",
    regexp_replace(lower(col("Text")), "<[^>]*>|/>|<br|[^a-z ]", " "))

# 2. Tokenizar
tok = Tokenizer(inputCol="Text", outputCol="palabras")
neg_tok = tok.transform(neg_txt)

# 3. Quitar SOLO stopwords estándar
sw_default = StopWordsRemover.loadDefaultStopWords("english")
# Mantenemos las negaciones aunque estén en la lista estándar
negaciones = ["no","not","never","dont","didnt","wont","cant","nothing"]
sw_lista = [w for w in sw_default if w not in negaciones]
sw = StopWordsRemover(inputCol="palabras", outputCol="limpias", stopWords=sw_lista)
neg_limpio = sw.transform(neg_tok)

# 4. Generar bigramas (pares de palabras consecutivas)
ngram = NGram(n=2, inputCol="limpias", outputCol="bigramas")
neg_bi = ngram.transform(neg_limpio)

# 5. Contar los bigramas más frecuentes
bigramas = (neg_bi.select(explode("bigramas").alias("bigrama"))
    .filter(length(col("bigrama")) > 6)   # quita pares muy cortos/ruido
    .groupBy("bigrama").agg(count("*").alias("frecuencia"))
    .orderBy(col("frecuencia").desc()))

print("Top 30 BIGRAMAS en reseñas NEGATIVAS:")
bigramas.show(30, truncate=False)

# Guardar
bigramas.limit(50).write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_bigramas_negativas")

Top 30 BIGRAMAS en reseñas NEGATIVAS:
+--------------+----------+
|bigrama       |frecuencia|
+--------------+----------+
|product       |9704      |
|flavor        |8570      |
|coffee        |7000      |
| however      |6050      |
| really       |5792      |
|however       |5037      |
| thought      |4591      |
| product      |4377      |
|amazon        |3882      |
|taste like    |3513      |
| bought       |3208      |
|tastes like   |3143      |
|ingredients   |3045      |
| tastes       |3016      |
| disappointed |3005      |
|not good      |2831      |
| flavor       |2814      |
| amazon       |2799      |
|not buy       |2786      |
|products      |2679      |
|better        |2640      |
| unfortunately|2604      |
| ordered      |2442      |
| coffee       |2427      |
|disappointed  |2374      |
|waste money   |2347      |
| tasted       |2292      |
|reviews       |2151      |
|not sure      |2132      |
|dog food      |2055      |
+--------------+----------+
only showi

In [0]:
from pyspark.sql.functions import col, trim, size, split
# LImpieza de bigramas para el dashboard
# Quedarnos solo con bigramas de 2 palabras reales (sin el ruido de espacios)
bigramas = spark.table("proyecto.sentimiento.gold_bigramas_negativas")
bigramas_limpio = (bigramas
    .withColumn("bigrama", trim(col("bigrama")))
    .filter(size(split(col("bigrama"), " ")) == 2)   # exactamente 2 palabras
    .orderBy(col("frecuencia").desc()))

bigramas_limpio.write.mode("overwrite").saveAsTable("proyecto.sentimiento.gold_bigramas_negativas")
bigramas_limpio.show(15, truncate=False)

+-------------+----------+
|bigrama      |frecuencia|
+-------------+----------+
|taste like   |3513      |
|tastes like  |3143      |
|not good     |2831      |
|not buy      |2786      |
|waste money  |2347      |
|not sure     |2132      |
|dog food     |2055      |
|not like     |1880      |
|much better  |1787      |
|not recommend|1769      |
|not worth    |1762      |
|not even     |1759      |
|gluten free  |1687      |
|grocery store|1644      |
+-------------+----------+

